# Capítulo 2: Describiendo lo que Ves (Estadística Descriptiva y Visual)

## Metáfora: Estadística como lentes de aumentos para datos

Antes de cualquier modelo o predicción, necesitamos **palpitar a nuestros datos**. Este notebook es el "Ojo Humano" — el análisis exploratorio que todo científico de datos debe hacer antes de sacar conclusiones.

**Reglas del juego:**
1. Usamos datos reales, sucios y actuales
2. Código con estructura: Configuración → Proceso → Visualización → Explicación
3. Ética siempre presente
4. Voz de científico de datos escéptico

## Celda 1: Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('Librerías importadas correctamente')

## Celda 2: Carga del Dataset

In [ ]:
# Cargar dataset de ventas
df = pd.read_csv('../datos/datos_ventas_superstore.csv', parse_dates=['OrderDate'])

# Primer vistazo
print(f'Dimensiones del dataset: {df.shape}')
print(f'\nPrimeras 5 filas:')
df.head()

In [ ]:
# Información general del dataset
print('\n=== INFORMACIÓN DEL DATASET ===')
print(f'Tipo de datos:')
print(df.dtypes)
print(f'\nValores nulos por columna:')
print(df.isnull().sum())
print(f'\nEstadísticas de fechas:')
print(f'Fecha mínima: {df["OrderDate"].min()}')
print(f'Fecha máxima: {df["OrderDate"].max()}')

## Celda 3: Estadísticas Resumidas

Aquí es donde **palpamos al paciente**. Media, mediana, desviación estándar y cuartiles nos cuentan la historia básica de nuestros datos.

In [ ]:
# Resumen estadístico de variables numéricas
print('=== RESUMEN ESTADÍSTICO ===')
resumen = df[['Sales', 'Profit', 'Quantity', 'Discount']].describe()
print(resumen.round(2))

In [ ]:
# Análisis detallado de cada variable
print('=== ANÁLISIS DETALLADO ===')

variables = ['Sales', 'Profit', 'Quantity', 'Discount']

for var in variables:
    print(f'\n--- {var.upper()} ---')
    print(f'  Media: ${df[var].mean():,.2f}')
    print(f'  Mediana: ${df[var].median():,.2f}')
    print(f'  Desviación estándar: ${df[var].std():,.2f}')
    print(f'  Mínimo: ${df[var].min():,.2f}')
    print(f'  Máximo: ${df[var].max():,.2f}')
    print(f'  Coeficiente de variación: {(df[var].std()/df[var].mean()*100):.1f}%')
    
    # Detectar sesgo
    skewness = df[var].skew()
    if skewness > 0.5:
        print(f'  Sesgo: Positivo ({skewness:.2f}) - Cola hacia la derecha')
    elif skewness < -0.5:
        print(f'  Sesgo: Negativo ({skewness:.2f}) - Cola hacia la izquierda')
    else:
        print(f'  Sesgo: Aproximadamente simétrico ({skewness:.2f})')

## Celda 4: Distribuciones con Histogramas

Los histogramas son las **huellas dactilares** de nuestros datos. Nos muestran dónde se concentran los valores.

In [ ]:
# Histogramas de variables numéricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, var in enumerate(variables):
    ax = axes[i]
    
    # Histograma
    sns.histplot(df[var], bins=25, kde=True, ax=ax, color='steelblue', alpha=0.7)
    
    # Líneas de media y mediana
    ax.axvline(df[var].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df[var].mean():,.0f}')
    ax.axvline(df[var].median(), color='green', linestyle='--', linewidth=2, label=f'Mediana: {df[var].median():,.0f}')
    
    ax.set_title(f'Distribución de {var}')
    ax.set_xlabel(var)
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.suptitle('Histogramas: Huellas Dactilares de los Datos', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Explicación de los Histogramas

Lo que buscamos:
- **Simetría:** ¿La distribución es simétrica o está sesgada?
- **Colas:** ¿Hay colas largas? (Indica valores extremos)
- **Media vs Mediana:** ¿Están cerca? Si no, hay sesgo
- **Múltiples picos:** ¿Hay subgrupos en los datos?

## Celda 5: Box Plots para Outliers

El box plot es la **radiografía** de nuestros datos. Los outliers son como las fracturas — hay que detectarlas.

In [ ]:
# Box plots de variables numéricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, var in enumerate(variables):
    ax = axes[i]
    
    bp = ax.boxplot(df[var].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'),
                    medianprops=dict(color='red', linewidth=2))
    
    ax.set_title(f'Distribución de {var}')
    ax.set_ylabel(var)
    
    # Calcular y mostrar outliers
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[var] < Q1 - 1.5 * IQR) | (df[var] > Q3 + 1.5 * IQR)]
    ax.text(1.1, df[var].max() * 0.9, f'Outliers: {len(outliers)}',
            fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Box Plots: Radiografía de los Datos', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Función para detectar outliers con IQR
def detectar_outliers(dataframe, columna):
    """
    Detecta outliers usando el método IQR
    """
    Q1 = dataframe[columna].quantile(0.25)
    Q3 = dataframe[columna].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = dataframe[(dataframe[columna] < limite_inferior) | 
                         (dataframe[columna] > limite_superior)]
    
    return outliers, limite_inferior, limite_superior

# Detectar outliers en Profit
outliers_profit, li, ls = detectar_outliers(df, 'Profit')
print(f'=== ANÁLISIS DE OUTLIERS EN BENEFICIO ===')
print(f'Límite inferior: ${li:,.2f}')
print(f'Límite superior: ${ls:,.2f}')
print(f'Número de outliers: {len(outliers_profit)}')
print(f'\nPrimeros 10 outliers:')
print(outliers_profit[['OrderDate', 'Region', 'Category', 'Sales', 'Profit']].head(10).to_string())

## Celda 6: Matriz de Correlación con Heatmap

La correlación es como **dos amigos que siempre caminan juntos**. Si uno sube, ¿el otro también sube?

In [ ]:
# Matriz de correlación
corr_matrix = df[['Sales', 'Profit', 'Quantity', 'Discount']].corr()

# Visualización con heatmap
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu_r', center=0,
            square=True, linewidths=0.5, fmt='.2f',
            mask=mask, vmin=-1, vmax=1)
plt.title('Matriz de Correlación: ¿Qué Variables Caminan Juntas?', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Interpretación de correlaciones
print('=== INTERPRETACIÓN DE CORRELACIONES ===')
print('\nRegla de interpretación:')
print('  0.7 - 1.0: Correlación fuerte positiva')
print('  0.4 - 0.7: Correlación moderada positiva')
print('  0.0 - 0.4: Correlación débil o nula')
print(' -0.4 - 0.0: Correlación débil negativa')
print(' -0.7 - -0.4: Correlación moderada negativa')
print(' -1.0 - -0.7: Correlación fuerte negativa')

print('\nCorrelaciones observadas:')
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr = corr_matrix.iloc[i, j]
        var1 = corr_matrix.columns[i]
        var2 = corr_matrix.columns[j]
        
        if abs(corr) > 0.7:
            fuerza = 'FUERTE'
        elif abs(corr) > 0.4:
            fuerza = 'MODERADA'
        else:
            fuerza = 'DÉBIL'
        
        print(f'  {var1} vs {var2}: {corr:.2f} ({fuerza})')

## Celda 7: Gráficos de Dispersión

Los gráficos de dispersión nos muestran la **relación entre dos amigos** — ¿realmente caminan juntos?

In [ ]:
# Scatter plots: Relación entre variables clave
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sales vs Profit
sns.scatterplot(data=df, x='Sales', y='Profit', hue='Category', 
                alpha=0.7, s=80, ax=axes[0])
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Ventas vs Beneficio')
axes[0].set_xlabel('Ventas ($)')
axes[0].set_ylabel('Beneficio ($)')

# Sales vs Quantity
sns.scatterplot(data=df, x='Quantity', y='Sales', hue='Category',
                alpha=0.7, s=80, ax=axes[1])
axes[1].set_title('Cantidad vs Ventas')
axes[1].set_xlabel('Cantidad')
axes[1].set_ylabel('Ventas ($)')

# Discount vs Profit
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category',
                alpha=0.7, s=80, ax=axes[2])
axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('Descuento vs Beneficio')
axes[2].set_xlabel('Descuento')
axes[2].set_ylabel('Beneficio ($)')

plt.suptitle('Relaciones entre Variables Clave', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Celda 8: Análisis por Segmentos

Los patrones más interesantes suelen estar escondidos en los segmentos. **Parte el dataset en pedazos** y mira cada uno.

In [ ]:
# Análisis por Región
print('=== ANÁLISIS POR REGIÓN ===')
resumen_region = df.groupby('Region').agg({
    'Sales': ['mean', 'median', 'sum', 'count'],
    'Profit': ['mean', 'sum'],
    'Discount': 'mean'
}).round(2)
print(resumen_region)

In [ ]:
# Análisis por Categoría
print('\n=== ANÁLISIS POR CATEGORÍA ===')
resumen_categoria = df.groupby('Category').agg({
    'Sales': ['mean', 'median', 'sum', 'count'],
    'Profit': ['mean', 'sum'],
    'Discount': 'mean'
}).round(2)
print(resumen_categoria)

In [ ]:
# Análisis combinado: Región + Categoría
print('\n=== ANÁLISIS COMBINADO: REGIÓN + CATEGORÍA ===')
resumen_combinado = df.groupby(['Region', 'Category']).agg({
    'Sales': 'mean',
    'Profit': 'mean',
    'Discount': 'mean'
}).round(2)
print(resumen_combinado)

In [ ]:
# Visualización de análisis por segmentos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Ventas promedio por región
df.groupby('Region')['Sales'].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Ventas Promedio por Región')
axes[0, 0].set_xlabel('Ventas ($)')

# 2. Beneficio promedio por categoría
df.groupby('Category')['Profit'].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Beneficio Promedio por Categoría')
axes[0, 1].set_xlabel('Beneficio ($)')
axes[0, 1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

# 3. Distribución de ventas por región (boxplot)
sns.boxplot(data=df, x='Region', y='Sales', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Distribución de Ventas por Región')

# 4. Descuento promedio por categoría
df.groupby('Category')['Discount'].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[1, 1], color='green')
axes[1, 1].set_title('Descuento Promedio por Categoría')
axes[1, 1].set_xlabel('Descuento')

plt.suptitle('Análisis por Segmentos: ¿Qué Nos Cuentan los Datos?', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Análisis temporal
print('=== ANÁLISIS TEMPORAL ===')

# Crear columna de mes
df['Month'] = df['OrderDate'].dt.to_period('M')

# Ventas mensuales
ventas_mensuales = df.groupby('Month').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'OrderDate': 'count'
}).rename(columns={'OrderDate': 'NumOrdenes'})

print(ventas_mensuales)

In [ ]:
# Visualización temporal
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Tendencia de ventas
ventas_mensuales['Sales'].plot(kind='line', marker='o', ax=axes[0], 
                              color='steelblue', linewidth=2)
axes[0].set_title('Tendencia de Ventas Mensuales', fontsize=14)
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('Ventas Totales ($)')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Tendencia de beneficio
ventas_mensuales['Profit'].plot(kind='line', marker='s', ax=axes[1], 
                                color='coral', linewidth=2)
axes[1].set_title('Tendencia de Beneficio Mensual', fontsize=14)
axes[1].set_xlabel('Mes')
axes[1].set_ylabel('Beneficio Total ($)')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Celda 9: Hallazgos y Conclusiones

### Resumen del Análisis Exploratorio

Después de este análisis, deberías ser capaz de responder:

1. **¿Cuál es la "típica" venta?** → La media y mediana nos dan la respuesta
2. **¿Qué regiones son más rentables?** → No solo las que más venden
3. **¿Hay categorías que siempre pierden dinero?** → Analizar Profit negativo
4. **¿El descuento afecta al beneficio?** → Correlación Discount-Profit
5. **¿Hay estacionalidad?** → Tendencias temporales

In [ ]:
# Resumen final de hallazgos
print('='*60)
print('HALLAZGOS PRINCIPALES DEL ANÁLISIS EXPLORATORIO')
print('='*60)

print(f'\n1. RESUMEN GENERAL:')
print(f'   - Total de órdenes: {len(df)}')
print(f'   - Período: {df["OrderDate"].min().strftime("%Y-%m-%d")} a {df["OrderDate"].max().strftime("%Y-%m-%d")}')
print(f'   - Ventas totales: ${df["Sales"].sum():,.2f}')
print(f'   - Beneficio total: ${df["Profit"].sum():,.2f}')
print(f'   - Margen de beneficio: {(df["Profit"].sum()/df["Sales"].sum()*100):.1f}%')

print(f'\n2. DISTRIBUCIÓN DE VENTAS:')
print(f'   - Media: ${df["Sales"].mean():,.2f}')
print(f'   - Mediana: ${df["Sales"].median():,.2f}')
print(f'   - Desviación estándar: ${df["Sales"].std():,.2f}')
print(f'   - Coeficiente de variación: {(df["Sales"].std()/df["Sales"].mean()*100):.1f}%')

print(f'\n3. OUTLIERS DETECTADOS:')
for var in variables:
    outliers, _, _ = detectar_outliers(df, var)
    print(f'   - {var}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')

print(f'\n4. CORRELACIONES CLAVE:')
print(f'   - Sales vs Profit: {corr_matrix.loc["Sales", "Profit"]:.2f}')
print(f'   - Discount vs Profit: {corr_matrix.loc["Discount", "Profit"]:.2f}')
print(f'   - Sales vs Quantity: {corr_matrix.loc["Sales", "Quantity"]:.2f}')

print(f'\n5. MEJORES SEGMENTOS:')
mejor_region = df.groupby('Region')['Profit'].mean().idxmax()
mejor_categoria = df.groupby('Category')['Profit'].mean().idxmax()
print(f'   - Región más rentable: {mejor_region}')
print(f'   - Categoría más rentable: {mejor_categoria}')

print(f'\n6. ALERTAS ÉTICAS:')
print(f'   - Verificar si hay sesgos en la representación de regiones/categorías')
print(f'   - Analizar outliers: ¿son errores o información valiosa?')
print(f'   - Considerar el impacto de los descuentos en la rentabilidad')

print('\n' + '='*60)
print('FIN DEL ANÁLISIS EXPLORATORIO')
print('='*60)

## Checklist Ético del Análisis

Antes de continuar, asegúrate de haber respondido:

- [ ] ¿La media es representativa o hay outliers que la distorsionan?
- [ ] ¿Las correlaciones tienen sentido causal o son coincidencia?
- [ ] ¿Las visualizaciones son honestas con las escalas?
- [ ] ¿Estoy considerando sesgos en la recolección de datos?
- [ ] ¿Los segmentos que estoy analando son estadísticamente significativos?

---

### Referencias

- Tukey, J. W. (1977). *Exploratory Data Analysis*. Addison-Wesley.
- Cleveland, W. S. (1993). *Visualizing Data*. Hobart Press.
- Tufte, E. R. (2001). *The Visual Display of Quantitative Information*. Graphics Press.
- Few, S. (2012). *Show Me the Numbers*. Analytics Press.

---

*Siguiente capítulo: Capítulo 3 — Limpieza de Datos: El Arte de Quitar la Basura*